<a href="https://colab.research.google.com/github/Rahid-Khan/FAG-chatbot/blob/main/work%20/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rahid-Khan/internship-work-/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

**My lane (locked):** content-refresh prioritization — predict which content items are at risk of **search-click decline**. Label: `is_declining = (trend_direction == 'down')`.

This is the optional-stretch leakage/privacy check whose core now lives in the ML-04 data contract. It builds an honest feature vector, then **attacks its own features** to prove no answer sneaks in.

> **Runs fully offline — no token.** Uses the committed slice `data/raw/content_refresh_anonymized.csv` (30k rows), so *Runtime → Run all* works with nothing to configure.
>
> **The leakage rule that governs everything below:** the label compares the **last 30 days vs the prior 30 days**. So the prediction moment sits at the boundary — 30 days ago. Only columns knowable at that moment are legal features: the **prev-30-day** window and **static** content/keyword properties. Anything from the **last 30 days** (or a 90-day window that contains them, or `trend_*` which defines the label) is off-limits.

## 0. Setup — load the committed slice (offline)

In [2]:
import pandas as pd, numpy as np
from pathlib import Path

CSV = None
for base in [Path.cwd(), *Path.cwd().parents]:
    cand = base / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    if cand.exists():
        CSV = cand
        break
if CSV is None:
    for cand in Path('/content').rglob('content_refresh_anonymized.csv'):
        CSV = cand
        break
assert CSV is not None, 'Could not find data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(CSV)
df['y_declined'] = (df['trend_direction'] == 'down').astype(int)
BASE_RATE = float(df['y_declined'].mean())
print('loaded', df.shape, 'from', CSV.name)
print(f'label base rate (declining) = {BASE_RATE:.3f}')

loaded (30000, 45) from content_refresh_anonymized.csv
label base rate (declining) = 0.542


## 1. Build the feature vector

Honest features only — everything here is knowable **before** the last-30-day label window:

- **Prior-window traffic** (`*_prev_30d`, days 31–60 back): the only traffic window strictly before the label. `log1p` because traffic is heavy-tailed.
- **Prior-window CTR**: `clicks_prev_30d / impressions_prev_30d` — derived only from the safe window.
- **Static content properties**: `content_age_days`, `days_since_last_update`, `word_count`, `char_count` (+ a `has_wordcount` flag, since word/char are missing for ~7.7k rows).
- **Keyword context**: `search_volume`, `competition`, `cpc` (+ a `has_keyword` flag — missingness follows `content_type`, so a blind `fillna(0)` would inject a content-type signal).
- **Categoricals** (one-hot): `content_type`, `main_intent`, `competition_level`.

Missing numerics → 0 **with a `has_` flag**; missing categoricals → `'unknown'`. IDs are never features.

In [3]:
def build_features(frame):
    X = pd.DataFrame(index=frame.index)
    # prior-window traffic (days 31-60 back) - strictly before the label window; heavy-tailed -> log1p
    X['log_impr_prev30']   = np.log1p(frame['impressions_prev_30d'].fillna(0))
    X['log_clicks_prev30'] = np.log1p(frame['clicks_prev_30d'].fillna(0))
    X['log_sess_prev30']   = np.log1p(frame['sessions_prev_30d'].fillna(0))
    ip = frame['impressions_prev_30d'].fillna(0)
    X['ctr_prev30'] = np.where(ip > 0, frame['clicks_prev_30d'].fillna(0) / ip, 0.0)
    # static content properties (known at prediction time)
    X['content_age_days']       = frame['content_age_days'].fillna(0)
    X['days_since_last_update'] = frame['days_since_last_update'].fillna(0)
    X['word_count']             = frame['word_count'].fillna(0)
    X['char_count']             = frame['char_count'].fillna(0)
    X['has_wordcount']          = frame['word_count'].notna().astype(int)
    # keyword context (static) + missingness flag (missingness follows content_type)
    X['search_volume'] = frame['search_volume'].fillna(0)
    X['competition']   = frame['competition'].fillna(0)
    X['cpc']           = frame['cpc'].fillna(0)
    X['has_keyword']   = frame['search_volume'].notna().astype(int)
    # categoricals -> one-hot (fill unknown)
    cats = frame[['content_type', 'main_intent', 'competition_level']].fillna('unknown')
    X = pd.concat([X, pd.get_dummies(cats, prefix=['ctype', 'intent', 'comp'])], axis=1)
    return X

X = build_features(df)
y = df['y_declined']
groups = df['client_id']
print('feature matrix:', X.shape, '->', X.shape[1], 'columns')
print('feature names:', list(X.columns))

feature matrix: (30000, 25) -> 25 columns
feature names: ['log_impr_prev30', 'log_clicks_prev30', 'log_sess_prev30', 'ctr_prev30', 'content_age_days', 'days_since_last_update', 'word_count', 'char_count', 'has_wordcount', 'search_volume', 'competition', 'cpc', 'has_keyword', 'ctype_comparison article', 'ctype_feedly article', 'ctype_keyword article', 'intent_commercial', 'intent_informational', 'intent_navigational', 'intent_transactional', 'intent_unknown', 'comp_HIGH', 'comp_LOW', 'comp_MEDIUM', 'comp_unknown']


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature group | Meaning | Missing handling | Knowable before the label window? |
|---|---|---|---|
| `log_*_prev30`, `ctr_prev30` | Traffic in days 31–60 back | `fillna(0)` (blanks are true zeros here) | **Yes** — strictly before the last-30 window |
| `content_age_days`, `days_since_last_update` | Age / staleness of the page | `fillna(0)` | **Yes** — a property, not an outcome |
| `word_count`, `char_count` (+`has_wordcount`) | Article length | `fillna(0)` + flag (missing for ~7.7k rows) | **Yes** — static content property |
| `search_volume`, `competition`, `cpc` (+`has_keyword`) | Keyword context | `fillna(0)` + flag (missing follows `content_type`) | **Yes** — keyword metadata, static |
| `ctype_*`, `intent_*`, `comp_*` | Content type / intent / competition band | `'unknown'` category | **Yes** — static metadata |

The cell below proves the missingness point: keyword/word-count blanks are **not random** — they follow `content_type`, which is exactly why we add `has_` flags instead of a blind `fillna(0)`.

In [4]:
miss = (df.assign(keyword_missing=df['search_volume'].isna(),
                  wordcount_missing=df['word_count'].isna())
          .groupby('content_type')
          .agg(n=('content_id', 'size'),
               keyword_missing=('keyword_missing', 'mean'),
               wordcount_missing=('wordcount_missing', 'mean'))
          .round(3))
print('Missing-rate by content_type (why we use has_ flags, not a blind fillna):')
print(miss.to_string())
print()
print('Reading: missingness is systematic (one content_type is ~100% missing keyword data).')
print('A blind fillna(0) would silently encode content_type into the numeric features.')

Missing-rate by content_type (why we use has_ flags, not a blind fillna):
                        n  keyword_missing  wordcount_missing
content_type                                                 
comparison article    697            0.000              0.000
feedly article       2096            1.000              0.000
keyword article     27207            0.014              0.283

Reading: missingness is systematic (one content_type is ~100% missing keyword data).
A blind fillna(0) would silently encode content_type into the numeric features.


## 3. The leakage hunt

Attack the features. Three ways an answer sneaks in — I test each on THIS dataset, then evaluate honestly.

**Method:** train a plain logistic model, evaluate with **GroupKFold by `client_id`** (rows from one client share hidden character; a random split lets the model memorize the client and fake skill). Print the **base rate** next to every AUC. Then deliberately add a leaky feature and watch the score jump toward 1.0 — if it doesn't, the test harness itself is broken.

- **Label-derived:** `trend_pct` is the label's own source → adding it should rocket AUC to ~1.0.
- **Future/overlapping window:** `*_last_30d` sits inside the label window; with `*_prev_30d` already present, the two halves reconstruct the label → AUC inflates.
- **Grouped vs random split:** the gap measures how much memorization a random split was hiding.

> **Note on `impressions_90d`:** the w04 baseline uses it as a *magnitude* weight for ranking and argues it is not directional. For a trained classifier predicting the *direction* label I take the stricter line and keep all 90-day aggregates out of the honest set, because the 90-day window **contains** the last-30-day label window. Different tool, different bar.

In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GroupKFold, StratifiedKFold

def make_model():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

def auc_grouped(Xf, yf, gf, k=5):
    gkf = GroupKFold(n_splits=k)
    s = cross_val_score(make_model(), Xf, yf, cv=gkf, groups=gf, scoring='roc_auc')
    return float(s.mean()), float(s.std())

def auc_random(Xf, yf, k=5):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=0)
    s = cross_val_score(make_model(), Xf, yf, cv=skf, scoring='roc_auc')
    return float(s.mean()), float(s.std())

h_auc, h_sd = auc_grouped(X, y, groups)
print(f'base rate (declining)                 = {BASE_RATE:.3f}')
print(f'HONEST features, grouped-by-client AUC = {h_auc:.3f} ± {h_sd:.3f}')
print('(modestly above 0.5 is expected and honest - predicting direction from prior level + static props is hard)')

base rate (declining)                 = 0.542
HONEST features, grouped-by-client AUC = 0.698 ± 0.068
(modestly above 0.5 is expected and honest - predicting direction from prior level + static props is hard)


In [6]:
# LEAK TEST 1 - label-derived: add trend_pct (the label's own source).
Xa = X.copy()
Xa['LEAK_trend_pct'] = df['trend_pct'].fillna(0)
a_auc, _ = auc_grouped(Xa, y, groups)

# LEAK TEST 2 - future/overlapping window: add the last-30d columns (inside the label window).
Xb = X.copy()
Xb['LEAK_log_impr_last30']   = np.log1p(df['impressions_last_30d'].fillna(0))
Xb['LEAK_log_clicks_last30'] = np.log1p(df['clicks_last_30d'].fillna(0))
Xb['LEAK_log_sess_last30']   = np.log1p(df['sessions_last_30d'].fillna(0))
b_auc, _ = auc_grouped(Xb, y, groups)

print(f'HONEST features                          AUC = {h_auc:.3f}')
print(f'+ trend_pct  (label source)              AUC = {a_auc:.3f}   <- confession (jumps toward 1.0)')
print(f'+ *_last_30d (future/overlapping window) AUC = {b_auc:.3f}   <- reconstructs the label with prev_30d')
assert a_auc > h_auc + 0.10, 'harness broken: a known leak did NOT inflate the score'
print()
print('harness works: the deliberately-added leak inflated AUC as expected.')

HONEST features                          AUC = 0.698
+ trend_pct  (label source)              AUC = 0.999   <- confession (jumps toward 1.0)
+ *_last_30d (future/overlapping window) AUC = 0.999   <- reconstructs the label with prev_30d

harness works: the deliberately-added leak inflated AUC as expected.


In [7]:
# Grouped vs random split on the HONEST features — the gap is a memorization finding.
r_auc, r_sd = auc_random(X, y)
print(f'honest AUC  grouped-by-client = {h_auc:.3f} ± {h_sd:.3f}')
print(f'honest AUC  random 5-fold     = {r_auc:.3f} ± {r_sd:.3f}')
print(f'gap (random - grouped)        = {r_auc - h_auc:+.3f}')
print()
print('A positive gap = the random split was letting the model memorize per-client character.')
print('The grouped number is the honest one: does it work on a client it never saw?')

honest AUC  grouped-by-client = 0.698 ± 0.068
honest AUC  random 5-fold     = 0.724 ± 0.007
gap (random - grouped)        = +0.025

A positive gap = the random split was letting the model memorize per-client character.
The grouped number is the honest one: does it work on a client it never saw?


## 4. What I excluded and why

| Excluded column(s) | Why |
|---|---|
| `trend_direction`, `trend_pct` | **The label and its source.** Using them is circular — this is the leak the whole check exists to prevent. |
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | **Inside the label window.** With `*_prev_30d` present they reconstruct the label exactly. |
| `impressions_90d`, `clicks_90d`, `sessions_90d`, `pageviews_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_*` | **90-day window contains the last-30 label window** — overlapping-window leakage for a direction label. |
| `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | Rates computed over the 90-day window → same overlap problem. |
| `impression_tier`, `position_tier` | Derived from the overlapping 90-day aggregates → inherit the overlap. |
| `content_id`, `client_id` | **Pseudonymous IDs** — grouping/splitting only, never features (would memorize identity). |
| `provider_used`, `model_used` | Generation metadata the data dictionary marks *not a model feature*. |

The cell below is a hard guard: it asserts none of the banned columns leaked into the feature matrix.

In [8]:
BANNED = {
    'trend_direction', 'trend_pct', 'is_declining_label', 'y_declined',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'pageviews_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'impression_tier', 'position_tier',
    'content_id', 'client_id', 'provider_used', 'model_used',
}
leaked = [c for c in X.columns if c in BANNED]
assert not leaked, f'LEAK: banned columns found in features -> {leaked}'
print('leakage guard PASSED - no banned label/future-window/id/flag column is a feature.')
print(f'honest feature count: {X.shape[1]}')
print('privacy: no client names, URLs, or raw queries printed anywhere; IDs used for grouping only.')

leakage guard PASSED - no banned label/future-window/id/flag column is a feature.
honest feature count: 25
privacy: no client names, URLs, or raw queries printed anywhere; IDs used for grouping only.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Leakage attacked three ways (label-derived, future window, grouped-vs-random) with a harness self-test and a hard guard assertion
- [ ] **Run top to bottom**, then commit under `work/notebooks/` and submit the repo URL